# 02 - Create the SQLite database and run first SQL queries

**Steps:** 1) Create the database file  2) Create the tables with keys  3) Load the clean data  4) Check the row counts  5) Run the first 8 queries

## 1. Import libraries and connect
`sqlite3` comes with Python. If the file `hr_analytics.db` does not exist, SQLite creates it.

In [ ]:
import sqlite3
import pandas as pd

DB_PATH = "../data/hr_analytics.db"
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")   # make SQLite enforce the links between tables
print("Connected to", DB_PATH)

## 2. Create the tables
- I renamed `DistanceFromHome (KM)` to `DistanceFromHome_KM`
- Dates are stored as text in `YYYY-MM-DD` format, because SQLite has no separate date type. This format still sorts correctly.

In [ ]:
schema = """
DROP TABLE IF EXISTS performance;
DROP TABLE IF EXISTS employee;
DROP TABLE IF EXISTS education_level;
DROP TABLE IF EXISTS rating_level;
DROP TABLE IF EXISTS satisfied_level;

CREATE TABLE education_level (
    EducationLevelID INTEGER PRIMARY KEY,
    EducationLevel   TEXT NOT NULL
);

CREATE TABLE rating_level (
    RatingID    INTEGER PRIMARY KEY,
    RatingLevel TEXT NOT NULL
);

CREATE TABLE satisfied_level (
    SatisfactionID    INTEGER PRIMARY KEY,
    SatisfactionLevel TEXT NOT NULL
);

CREATE TABLE employee (
    EmployeeID              TEXT PRIMARY KEY,
    FirstName               TEXT,
    LastName                TEXT,
    Gender                  TEXT,
    Age                     INTEGER,
    BusinessTravel          TEXT,
    Department              TEXT,
    DistanceFromHome_KM     INTEGER,
    State                   TEXT,
    Ethnicity               TEXT,
    Education               INTEGER,
    EducationField          TEXT,
    JobRole                 TEXT,
    MaritalStatus           TEXT,
    Salary                  INTEGER,
    StockOptionLevel        INTEGER,
    OverTime                TEXT,
    HireDate                TEXT,
    Attrition               TEXT,
    YearsAtCompany          INTEGER,
    YearsInMostRecentRole   INTEGER,
    YearsSinceLastPromotion INTEGER,
    YearsWithCurrManager    INTEGER,
    FOREIGN KEY (Education) REFERENCES education_level (EducationLevelID)
);

CREATE TABLE performance (
    PerformanceID                   TEXT PRIMARY KEY,
    EmployeeID                      TEXT NOT NULL,
    ReviewDate                      TEXT,
    EnvironmentSatisfaction         INTEGER,
    JobSatisfaction                 INTEGER,
    RelationshipSatisfaction        INTEGER,
    TrainingOpportunitiesWithinYear INTEGER,
    TrainingOpportunitiesTaken      INTEGER,
    WorkLifeBalance                 INTEGER,
    SelfRating                      INTEGER,
    ManagerRating                   INTEGER,
    FOREIGN KEY (EmployeeID)               REFERENCES employee (EmployeeID),
    FOREIGN KEY (EnvironmentSatisfaction)  REFERENCES satisfied_level (SatisfactionID),
    FOREIGN KEY (JobSatisfaction)          REFERENCES satisfied_level (SatisfactionID),
    FOREIGN KEY (RelationshipSatisfaction) REFERENCES satisfied_level (SatisfactionID),
    FOREIGN KEY (SelfRating)               REFERENCES rating_level (RatingID),
    FOREIGN KEY (ManagerRating)            REFERENCES rating_level (RatingID)
);
"""
conn.executescript(schema)
print("Tables created")

Tables created


## 3. Load the clean CSV files into the tables
The order matters: the lookup tables first, then `employee`, then `performance`. A foreign key can only point to a row that already exists.

In [ ]:
education_level = pd.read_csv("../data/clean/education_level_clean.csv")
rating_level    = pd.read_csv("../data/clean/rating_level_clean.csv")
satisfied_level = pd.read_csv("../data/clean/satisfied_level_clean.csv")
employee        = pd.read_csv("../data/clean/employee_clean.csv")
performance     = pd.read_csv("../data/clean/performance_clean.csv")

# same rename we used in the CREATE TABLE statement
employee = employee.rename(columns={"DistanceFromHome (KM)": "DistanceFromHome_KM"})

education_level.to_sql("education_level", conn, if_exists="append", index=False)
rating_level.to_sql("rating_level", conn, if_exists="append", index=False)
satisfied_level.to_sql("satisfied_level", conn, if_exists="append", index=False)
employee.to_sql("employee", conn, if_exists="append", index=False)
performance.to_sql("performance", conn, if_exists="append", index=False)
conn.commit()
print("Data loaded")

Data loaded


## 4. Check that everything was loaded
The numbers must match what we saw in notebook 01.

In [ ]:
for table in ["employee", "performance", "education_level", "rating_level", "satisfied_level"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {n} rows")

problems = conn.execute("PRAGMA foreign_key_check").fetchall()
print("Foreign key problems:", len(problems))

employee: 1470 rows
performance: 6709 rows
education_level: 5 rows
rating_level: 5 rows
satisfied_level: 5 rows
Foreign key problems: 0


## 5. Helper function
`run(sql)` sends a query to the database and shows the result as a table.

In [ ]:
def run(sql):
    return pd.read_sql_query(sql, conn)

## 6. Data quality checks on the date columns
Dates are stored as text in `YYYY-MM-DD` format. This is the standard way in SQLite, because text in this format sorts in date order and works with SQLite date functions. These checks prove it.

In [ ]:
run("""
SELECT 'employee.HireDate'      AS column_name, typeof(HireDate)   AS stored_as, COUNT(*) AS rows_count,
       MIN(HireDate)   AS first_date, MAX(HireDate)   AS last_date,
       SUM(date(HireDate) IS NULL) AS invalid_dates
FROM employee GROUP BY typeof(HireDate)
UNION ALL
SELECT 'performance.ReviewDate', typeof(ReviewDate), COUNT(*),
       MIN(ReviewDate), MAX(ReviewDate),
       SUM(date(ReviewDate) IS NULL)
FROM performance GROUP BY typeof(ReviewDate);
""")

,column_name,stored_as,rows_count,first_date,last_date,invalid_dates
0,employee.HireDate,text,1470,2012-01-03,2022-12-31,0
1,performance.ReviewDate,text,6709,2013-01-02,2022-12-31,0


`strftime` extracts a part of the date. Example: how many employees were hired in each year.

In [ ]:
run("""
SELECT strftime('%Y', HireDate) AS hire_year, COUNT(*) AS employees
FROM employee
GROUP BY hire_year
ORDER BY hire_year;
""")

,hire_year,employees
0,2012,151
1,2013,136
2,2014,136
3,2015,127
4,2016,114
5,2017,106
6,2018,136
7,2019,145
8,2020,127
9,2021,137


**Logic check:** a performance review should not be dated before the employee was hired. Let us count how many reviews break this rule.

In [ ]:
run("""
SELECT
    COUNT(*) AS total_reviews,
    SUM(p.ReviewDate < e.HireDate) AS reviews_before_hire,
    ROUND(100.0 * SUM(p.ReviewDate < e.HireDate) / COUNT(*), 1) AS pct_before_hire,
    COUNT(DISTINCT CASE WHEN p.ReviewDate < e.HireDate THEN e.EmployeeID END) AS employees_affected
FROM performance p
JOIN employee e ON p.EmployeeID = e.EmployeeID;
""")

,total_reviews,reviews_before_hire,pct_before_hire,employees_affected
0,6709,1141,17.0,289


**Note:** this is a data quality issue in the source data. I keep the rows (I do not delete data without a clear reason), and I will be careful when analysing trends over time. I will mention it in the README.

## 7. The first 8 queries
The same queries are saved in `sql/01_basic_queries.sql` for GitHub.

### 1. Look at the first rows

In [ ]:
run("""
SELECT EmployeeID, FirstName, LastName, Department, JobRole, Salary
FROM employee
LIMIT 5;
""")

,EmployeeID,FirstName,LastName,Department,JobRole,Salary
0,3012-1A41,Leonelle,Simco,Sales,Sales Executive,102059
1,CBCB-9C9D,Leonerd,Aland,Sales,Sales Executive,157718
2,95D7-1CE9,Ahmed,Sykes,Human Resources,HR Business Partner,309964
3,47A0-559B,Ermentrude,Berrie,Technology,Engineering Manager,293132
4,42CC-040A,Stace,Savege,Human Resources,Recruiter,49606


### 2. Overall attrition rate

In [ ]:
run("""
SELECT
    COUNT(*) AS total_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS employees_left,
    ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
FROM employee;
""")

,total_employees,employees_left,attrition_rate_pct
0,1470,237,16.1


### 3. Attrition rate by department

In [ ]:
run("""
SELECT
    Department,
    COUNT(*) AS total_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS employees_left,
    ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
FROM employee
GROUP BY Department
ORDER BY attrition_rate_pct DESC;
""")

,Department,total_employees,employees_left,attrition_rate_pct
0,Sales,446,92,20.6
1,Human Resources,63,12,19.0
2,Technology,961,133,13.8


### 4. Does overtime change the attrition rate?
Same idea as query 3, but we group by `OverTime` (Yes / No).

In [ ]:
run("""
SELECT
    OverTime,
    COUNT(*) AS total_employees,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS employees_left,
    ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS attrition_rate_pct
FROM employee
GROUP BY OverTime;
""")

,OverTime,total_employees,employees_left,attrition_rate_pct
0,No,1054,110,10.4
1,Yes,416,127,30.5


### 5. Average salary: employees who left vs stayed

In [ ]:
run("""
SELECT
    Attrition,
    COUNT(*) AS employees,
    ROUND(AVG(Salary)) AS avg_salary
FROM employee
GROUP BY Attrition;
""")

,Attrition,employees,avg_salary
0,No,1233,118856.0
1,Yes,237,82262.0


### 6. Employees by education level
The employee table only has a number (1 to 5) in `Education`.

In [ ]:
run("""
SELECT
    el.EducationLevel AS education,
    COUNT(*) AS employees
FROM employee e
JOIN education_level el ON e.Education = el.EducationLevelID
GROUP BY el.EducationLevelID, el.EducationLevel
ORDER BY el.EducationLevelID;
""")

,education,employees
0,No Formal Qualifications,170
1,High School,282
2,Bachelors,572
3,Masters,398
4,Doctorate,48


### 7. Employees with no performance review

In [ ]:
run("""
SELECT COUNT(*) AS employees_without_reviews
FROM employee e
LEFT JOIN performance p ON e.EmployeeID = p.EmployeeID
WHERE p.EmployeeID IS NULL;
""")

,employees_without_reviews
0,190


### 8. Self rating vs manager rating by department
Joins employees with their reviews, then compares the average self rating with the average manager rating in each department.

In [ ]:
run("""
SELECT
    e.Department,
    COUNT(*) AS reviews,
    ROUND(AVG(p.SelfRating), 2) AS avg_self_rating,
    ROUND(AVG(p.ManagerRating), 2) AS avg_manager_rating
FROM employee e
JOIN performance p ON e.EmployeeID = p.EmployeeID
GROUP BY e.Department
ORDER BY e.Department;
""")

,Department,reviews,avg_self_rating,avg_manager_rating
0,Human Resources,303,3.99,3.44
1,Sales,2149,3.97,3.45
2,Technology,4257,3.99,3.49


## 8. Close the connection
After this cell the connection is closed. To run more queries, run the connection cell (section 1) again.

In [ ]:
conn.close()

## My notes

**What I found**
- The overall attrition rate is 16.1% (237 of 1,470 employees left).
- Sales has the highest attrition rate (20.6%), followed by Human Resources (19.0%) and Technology (13.8%). HR is a small group (63 employees), so its rate can change a lot with just a few people.
- Employees who work overtime left much more often (30.5%) than employees who do not (10.4%).
- Employees who left earned less on average (82,262) than employees who stayed (118,856).
- Self ratings are higher than manager ratings in all three departments (about 3.97-3.99 vs 3.44-3.49).
- 190 employees have no performance reviews, so I used LEFT JOIN to keep them in the analysis.

**What I need to be careful about**
- These results show relationships, not causes. For example, overtime and attrition happen together, but this does not prove that overtime makes people leave.
- The salary gap may be caused by job role, department, or seniority and not only by attrition. I need to compare salaries inside the same job role.
- 17% of the reviews are dated before the employee's hire date, so I will be careful with any analysis over time.

**What I want to explore next**
- Attrition by job role, age group, and years since last promotion.
- Whether overtime still matters inside each department.
- Salary compared with job role and years at the company.
- The gap between self rating and manager rating, and whether it relates to attrition.